# Stage 2 — Timing pilot (P5)

**Purpose.** Measure how long one fit takes for each of the ten models, so the tuning budget and the split of work between Colab and the PC are based on measurements, not guesses. **No performance score is computed or displayed**: the notebook fits, predicts on the validation fold and keeps only the elapsed time (P5).

**Design (S7).** For each model, two boundary configurations of its approved search space (D5/P4) are timed, the cheapest and the most expensive. Each is fitted on development folds 1–4 and predicts fold 0, which is one tuning fold. KNN and MLP are tuned on the 50,000-row subsample (D14), so they are timed on that, and their most expensive configuration is also timed on a full development fold, because the common 5-fold re-evaluation (D7) runs on full data. The notebook also freezes the 30 candidate configurations per model that the Stage 3 search will draw, and saves them as CSV.

**Run on Colab:** files in `MyDrive/thesis_xai/`: `thesis_common.py`, `requirements.txt`, `data/`, `stage1_outputs/`. Then *Run all*. **Run on the PC:** same folder layout, Python 3.13 with `pip install -r requirements.txt`; set `THESIS_MACHINE=pc` or leave the default.

If the session disconnects, run all again: models already timed on this machine are skipped.

In [1]:
# ---------- 0. Environment and paths ----------
import os, sys, json, time, warnings, importlib, platform, datetime as dt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"
    MACHINE = "colab"
else:
    PROJECT_DIR = os.environ.get("THESIS_PROJECT_DIR", os.path.abspath("."))
    MACHINE = os.environ.get("THESIS_MACHINE", "pc")
SMOKE = os.environ.get("THESIS_SMOKE") == "1"   # code test on a small sample only; never used for planning
sys.path.insert(0, PROJECT_DIR)
import thesis_common as tc
importlib.reload(tc)

import numpy as np
import pandas as pd

OUT_DIR = os.path.join(PROJECT_DIR, "stage2_smoke" if SMOKE else "stage2_outputs")
CFG_DIR = os.path.join(PROJECT_DIR, "search_configs")
os.makedirs(OUT_DIR, exist_ok=True)
print("Machine:", MACHINE, "| Python", platform.python_version(), "| CPUs:", os.cpu_count(), "| smoke test:", SMOKE)

Machine: pc | Python 3.13.0 | CPUs: 12 | smoke test: False


In [2]:
# ---------- 1. Data, frozen indices, library pins ----------
X, y = tc.load_data(PROJECT_DIR)                 # checks the dataset SHA-256
idx = tc.load_indices(PROJECT_DIR)               # checks every index file against MANIFEST.csv
folds = idx["dev_folds.csv"]
sub = idx["dev_subsample_50k.csv"]
env = tc.environment(MACHINE)
pin_diff = tc.check_pins(PROJECT_DIR, env["packages"])
print("Differences from requirements.txt:", pin_diff or "none")
if pin_diff and not SMOKE:
    raise SystemExit("Installed versions differ from requirements.txt (S6). Install the pinned versions first.")

def fold0(frame):
    """Training rows = folds 1-4, validation rows = fold 0 (row_id positions in the raw file)."""
    tr = frame.row_id[frame.fold != 0].to_numpy()
    va = frame.row_id[frame.fold == 0].to_numpy()
    if SMOKE:
        rng = np.random.default_rng(0)
        tr, va = rng.choice(tr, 4000, replace=False), rng.choice(va, 1000, replace=False)
    return tr, va

SCOPES = {"full_fold": fold0(folds), "subsample_fold": fold0(sub)}
{k: (len(v[0]), len(v[1])) for k, v in SCOPES.items()}

Differences from requirements.txt: none


{'full_fold': (162355, 40589), 'subsample_fold': (40000, 10000)}

In [3]:
# ---------- 2. Freeze the Stage 3 candidate configurations (search_sampling seed) ----------
cfg_rows = []
for m in tc.MODELS:
    frame = tc.configs_frame(m)
    h, status = tc.write_or_verify(frame, os.path.join(CFG_DIR, f"search_configs_{m}.csv"))
    cfg_rows.append({"model": m, "n_configs": len(frame), "sha256": h, "status": status})
cfg_df = pd.DataFrame(cfg_rows)
cfg_df.to_csv(os.path.join(OUT_DIR, f"search_configs_check_{MACHINE}.csv"), index=False)
if cfg_df.status.str.startswith("MISMATCH").any():
    raise SystemExit("Sampled configurations differ from the frozen files. Check library versions.")
cfg_df

,model,n_configs,sha256,status
0,LR,30,6e57d88526c13b6d641b5ebc8f47c53b14108da1171283...,verified: identical to existing file
1,DT,30,5509962d584532ce3e94de9bc9997aa7d1d341d5c1ac22...,verified: identical to existing file
2,RF,30,a55fb3b68176458ad52249a6c61e0fcb54283367db99ce...,verified: identical to existing file
3,SVC,30,95b25eb50f13c8b9324dd76c61556d7897a50a97e78793...,verified: identical to existing file
4,KNN,30,52c566a8247103a66c437869aebe389c135859c1ad08d9...,verified: identical to existing file
5,GNB,30,937143e666e05b4e3c126ac189f141a9fc0fed44de0245...,verified: identical to existing file
6,XGB,30,9bae2e4ca16ccfba4c721bd460c2bd103879cb01886034...,verified: identical to existing file
7,LGBM,30,72f5b51f5d3087fff2a1cafc1c7fa0e9d544275142a0a9...,verified: identical to existing file
8,CAT,30,c0a1b9f3a9850d8e2f611e7035652ab788af624d7ccca6...,verified: identical to existing file
9,MLP,30,dce936e7e73119a57e04ba19a950c69777691d1883300c...,verified: identical to existing file


In [4]:
# ---------- 3. Timing runs (fit + predict only; no score is computed) ----------
from sklearn.exceptions import ConvergenceWarning

TIMING_CSV = os.path.join(OUT_DIR, f"pilot_timings_{MACHINE}.csv")
MODELS_TO_RUN = os.environ.get("THESIS_MODELS", ",".join(tc.MODELS)).split(",")

plan = []
for m in MODELS_TO_RUN:
    scope = "subsample_fold" if m in tc.SUBSAMPLE_TUNED else "full_fold"
    plan += [(m, "min", scope), (m, "max", scope)]
    if m in tc.SUBSAMPLE_TUNED:
        plan.append((m, "max", "full_fold"))     # cost of the D7 re-evaluation on full data

done = set()
if os.path.exists(TIMING_CSV):
    prev = pd.read_csv(TIMING_CSV)
    done = set(zip(prev.model, prev.config, prev.scope))

for m, level, scope in plan:
    if (m, level, scope) in done:
        print(f"skip {m} {level} {scope} (already timed)")
        continue
    tr, va = SCOPES[scope]
    pipe = tc.make_pipeline(m, n_jobs=-1).set_params(**tc.PILOT_CONFIGS[m][level])
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", ConvergenceWarning)
        t0 = time.perf_counter()
        pipe.fit(X.iloc[tr], y[tr])
        t1 = time.perf_counter()
        _ = tc.positive_score(pipe, X.iloc[va])   # timed and discarded; never scored
        t2 = time.perf_counter()
    clf = pipe.named_steps["clf"]
    row = {"model": m, "config": level, "scope": scope, "n_train": len(tr), "n_val": len(va),
           "fit_s": round(t1 - t0, 3), "predict_s": round(t2 - t1, 3), "total_s": round(t2 - t0, 3),
           "multithreaded": m in tc.MULTITHREADED,
           "convergence_warnings": sum(issubclass(x.category, ConvergenceWarning) for x in w),
           "mlp_epochs": getattr(clf, "n_iter_", None) if m == "MLP" else None,
           "machine": MACHINE, "cpu_count": os.cpu_count(),
           "run_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
    pd.DataFrame([row]).to_csv(TIMING_CSV, mode="a", header=not os.path.exists(TIMING_CSV), index=False)
    print(f"{m:5s} {level:3s} {scope:15s} fit {row['fit_s']:8.1f}s  predict {row['predict_s']:7.1f}s"
          + (f"  [{row['convergence_warnings']} convergence warning(s)]" if row["convergence_warnings"] else ""))

timings = pd.read_csv(TIMING_CSV)
timings

LR    min full_fold       fit      0.4s  predict     0.0s
LR    max full_fold       fit      5.8s  predict     0.0s
DT    min full_fold       fit      0.2s  predict     0.0s
DT    max full_fold       fit      0.8s  predict     0.0s
RF    min full_fold       fit      2.1s  predict     0.1s
RF    max full_fold       fit     23.0s  predict     0.4s
SVC   min full_fold       fit      0.6s  predict     0.0s
SVC   max full_fold       fit      8.4s  predict     0.0s
KNN   min subsample_fold  fit      0.0s  predict     0.7s
KNN   max subsample_fold  fit      0.0s  predict     2.4s
KNN   max full_fold       fit      0.1s  predict    34.5s
GNB   min full_fold       fit      0.1s  predict     0.0s
GNB   max full_fold       fit      0.1s  predict     0.0s
XGB   min full_fold       fit      0.6s  predict     0.0s
XGB   max full_fold       fit      4.7s  predict     0.1s
LGBM  min full_fold       fit      0.6s  predict     0.0s
LGBM  max full_fold       fit      5.5s  predict     0.3s
CAT   min full

,model,config,scope,n_train,n_val,fit_s,predict_s,total_s,multithreaded,convergence_warnings,mlp_epochs,machine,cpu_count,run_utc
0,LR,min,full_fold,162355,40589,0.401,0.018,0.419,False,0,NaN,pc,12,2026-10-11T19:05:06+00:00
1,LR,max,full_fold,162355,40589,5.805,0.016,5.821,False,0,NaN,pc,12,2026-10-11T19:05:11+00:00
2,DT,min,full_fold,162355,40589,0.195,0.010,0.205,False,0,NaN,pc,12,2026-10-11T19:05:12+00:00
3,DT,max,full_fold,162355,40589,0.823,0.016,0.839,False,0,NaN,pc,12,2026-10-11T19:05:13+00:00
4,RF,min,full_fold,162355,40589,2.127,0.071,2.198,True,0,NaN,pc,12,2026-10-11T19:05:15+00:00
5,RF,max,full_fold,162355,40589,22.965,0.359,23.324,True,0,NaN,pc,12,2026-10-11T19:05:39+00:00
6,SVC,min,full_fold,162355,40589,0.604,0.015,0.619,False,0,NaN,pc,12,2026-10-11T19:05:39+00:00
7,SVC,max,full_fold,162355,40589,8.435,0.020,8.455,False,0,NaN,pc,12,2026-10-11T19:05:48+00:00
8,KNN,min,subsample_fold,40000,10000,0.028,0.676,0.703,True,0,NaN,pc,12,2026-10-11T19:05:48+00:00
9,KNN,max,subsample_fold,40000,10000,0.023,2.410,2.433,True,0,NaN,pc,12,2026-10-11T19:05:51+00:00


In [5]:
# ---------- 4. Planning estimates (assumptions stated; these are not results) ----------
# Stage 3 search = 30 configurations x 5 folds = 150 fits. Mean cost per fit is approximated by the
# midpoint of the cheapest and most expensive configuration; the range uses the two extremes.
# Single-threaded models can run folds in parallel (search n_jobs = CPU count), so their wall time is
# divided by the CPU count; multithreaded models run folds one after another.
# Stage 4 (D7) = 5 fits of the frozen configuration on full development folds, bounded by the max config.
n_cfg = cfg_df.set_index("model").n_configs
est = []
for m in sorted(timings.model.unique(), key=tc.MODELS.index):
    t = timings[timings.model == m]
    tune_scope = "subsample_fold" if m in tc.SUBSAMPLE_TUNED else "full_fold"
    tmin = float(t[(t.config == "min") & (t.scope == tune_scope)].total_s.iloc[0])
    tmax = float(t[(t.config == "max") & (t.scope == tune_scope)].total_s.iloc[0])
    fits = int(n_cfg[m]) * tc.N_FOLDS
    par = 1 if m in tc.MULTITHREADED else os.cpu_count()
    full_max = float(t[(t.config == "max") & (t.scope == "full_fold")].total_s.iloc[0])
    est.append({"model": m, "fits_stage3": fits, "fit_s_min": tmin, "fit_s_max": tmax,
                "stage3_h_low": fits * tmin / par / 3600, "stage3_h_mid": fits * (tmin + tmax) / 2 / par / 3600,
                "stage3_h_high": fits * tmax / par / 3600,
                "stage4_h_upper": (tc.N_FOLDS * full_max + 1.25 * full_max) / 3600,
                "parallel_folds": par})
est_df = pd.DataFrame(est)
est_df.round(3).to_csv(os.path.join(OUT_DIR, f"pilot_estimates_{MACHINE}.csv"), index=False)
print(f"Total Stage 3 estimate on {MACHINE}: {est_df.stage3_h_mid.sum():.2f} h "
      f"(range {est_df.stage3_h_low.sum():.2f}-{est_df.stage3_h_high.sum():.2f} h); "
      f"Stage 4 upper bound {est_df.stage4_h_upper.sum():.2f} h")
est_df.round(3)

Total Stage 3 estimate on pc: 1.59 h (range 0.35-2.82 h); Stage 4 upper bound 0.23 h


,model,fits_stage3,fit_s_min,fit_s_max,stage3_h_low,stage3_h_mid,stage3_h_high,stage4_h_upper,parallel_folds
0,LR,150,0.419,5.821,0.001,0.011,0.020,0.010,12
1,DT,150,0.205,0.839,0.001,0.002,0.003,0.001,12
2,RF,150,2.198,23.324,0.092,0.532,0.972,0.040,1
3,SVC,150,0.619,8.455,0.002,0.016,0.029,0.015,12
4,KNN,150,0.703,2.433,0.029,0.065,0.101,0.060,1
5,GNB,150,0.093,0.091,0.000,0.000,0.000,0.000,12
6,XGB,150,0.662,4.778,0.028,0.113,0.199,0.008,1
7,LGBM,150,0.582,5.792,0.024,0.133,0.241,0.010,1
8,CAT,150,3.212,18.280,0.134,0.448,0.762,0.032,1
9,MLP,150,1.006,11.881,0.042,0.268,0.495,0.056,1


In [6]:
# ---------- 5. Environment log ----------
env.update({"stage": 2, "run_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "smoke_test": SMOKE, "pin_differences": pin_diff, "seeds_model": tc.MODEL_SEED,
            "search_sampling_seed": tc.SEEDS["search_sampling"], "config_version": tc.CONFIG_VERSION})
try:
    env["ram_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
except (ValueError, OSError, AttributeError):
    env["ram_gb"] = None   # not available on Windows; report RAM manually
with open(os.path.join(OUT_DIR, f"env_log_stage2_{MACHINE}.json"), "w") as f:
    json.dump(env, f, indent=2)
print("Send back:", TIMING_CSV, "and", os.path.join(OUT_DIR, f"pilot_estimates_{MACHINE}.csv"))

Send back: C:\Users\USER\Downloads\thesis_xai\stage2_outputs\pilot_timings_pc.csv and C:\Users\USER\Downloads\thesis_xai\stage2_outputs\pilot_estimates_pc.csv
